Example 5.12

In [ ]:

WORKSPACE_ID = "<your_workspace_id>"
LAKEHOUSE_ID = "<your_lakehouse_name>"

lakehouse_info = notebookutils.lakehouse.get(name=LAKEHOUSE_ID, workspaceId=WORKSPACE_ID)
abfss_path = lakehouse_info["properties"]["abfsPath"]
notebookutils.fs.mount(abfss_path, f"/{LAKEHOUSE_ID}") 
source_lakehouse = f"file://{notebookutils.fs.getMountPath(f"/{LAKEHOUSE_ID}")}"
df = spark.read.format("parquet").load(f"{source_lakehouse}/Files/customers")

Example 5.15

In [ ]:
from deltalake import DeltaTable, write_deltalake
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
entra_id_token = credential.get_token(
    "https://storage.azure.com/.default"
).token

storage_options = {
    "bearer_token": entra_id_token,
    "use_fabric_endpoint": "true",
}

WORKSPACE_ID = "1fe94944-3b5c-452c-9f37-fcf89c04d137"
LAKEHOUSE_ID = "b3903752-ede3-442b-a3a6-4170d826fc1f"
DELTA_TABLE_NAME = "customers"
DELTA_TABLE_PATH = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
    f"{LAKEHOUSE_ID}/Tables/{DELTA_TABLE_NAME}"
)

df_delta = DeltaTable(
    DELTA_TABLE_PATH,
    storage_options=storage_options,
)

df = df_delta.to_pandas()
write_deltalake(
    f"{DELTA_TABLE_PATH}_from_local",
    df,
    storage_options=storage_options,
)

Example 5.16

In [ ]:
import struct
from azure.identity import DefaultAzureCredential
import pyodbc
import pandas as pd

SERVER = "<your_server_name>"
DATABASE = "silver"
CONNECTION_STRING = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={SERVER};DATABASE={DATABASE};Encrypt=yes"

credential = DefaultAzureCredential()
token = credential.get_token("https://database.windows.net/.default")

def _pack_token(s: str) -> bytes:
    encoded = s.encode("utf-8")
    body = b"".join(bytes((b, 0)) for b in encoded)
    return struct.pack("=i", len(body)) + body

attrs_before={
    1256: _pack_token(token.token) # SQL_COPT_SS_ACCESS_TOKEN
}

with pyodbc.connect(CONNECTION_STRING, attrs_before=attrs_before) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM [silver].[dbo].[customers]")
    rows = cursor.fetchall()
    df = pd.DataFrame.from_records(rows, columns=[column[0] for column in cursor.description])
    print(df)